# LSTM architecture: the gates, step by step
1. One LSTM time step by hand with NumPy, checked against Keras.
2. Parameter count of an LSTM layer.
3. A small LSTM trained on IMDB reviews: the forget, input and output gates and the cell state, word by word,
   on a real review.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
DATA = Path("data")
DATA.mkdir(exist_ok=True)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

## 1. One time step by hand
2 units; the input is a one-hot word over the vocabulary cat, mat, rat (3 features).
Each weight matrix has 5 rows (h_1, h_2, cat, mat, rat) and 2 columns (one per unit). Biases are 0.

In [ ]:
W_f = np.array([[0.5, 0.0], [0.0, 0.5], [1.0, 0.0], [2.0, -1.0], [0.0, 1.0]])
W_i = np.array([[0.2, 0.0], [0.0, 0.2], [0.5, 0.5], [-1.0, 2.0], [0.0, 0.0]])
W_c = np.array([[0.1, 0.0], [0.0, 0.1], [0.3, 0.3], [0.5, -1.0], [0.0, 0.0]])
W_o = np.array([[0.3, 0.0], [0.0, 0.3], [0.0, 0.0], [1.0, 0.5], [0.0, 0.0]])
h_prev, c_prev = np.array([0.3, -0.2]), np.array([0.8, -0.5])
x_t = np.array([0.0, 1.0, 0.0])                  # "mat"

hx = np.concatenate([h_prev, x_t])               # [h_(t-1), x_t], 5 numbers
f = sigmoid(hx @ W_f)                            # forget gate
i = sigmoid(hx @ W_i)                            # input gate
c_tilde = np.tanh(hx @ W_c)                      # candidate cell state
o = sigmoid(hx @ W_o)                            # output gate
c = f * c_prev + i * c_tilde                     # new cell state
h = o * np.tanh(c)                               # new hidden state
for name, v in [("[h,x]W_f", hx @ W_f), ("f_t", f), ("[h,x]W_i", hx @ W_i), ("i_t", i), ("[h,x]W_c", hx @ W_c),
                ("c~_t", c_tilde), ("[h,x]W_o", hx @ W_o), ("o_t", o), ("f*c_prev", f * c_prev),
                ("i*c~", i * c_tilde), ("c_t", c), ("tanh(c_t)", np.tanh(c)), ("h_t", h)]:
    print(f"{name:10} {np.round(v, 3)}")

In [ ]:
# The same step in Keras. Keras keeps the x rows in `kernel` and the h rows in `recurrent_kernel`,
# with the four gates side by side in the order i, f, c, o.
lstm = keras.layers.LSTM(2, return_state=True)
x_in = x_t[None, None, :].astype("float32")      # (batch 1, 1 time step, 3 features)
init = [h_prev[None].astype("float32"), c_prev[None].astype("float32")]
lstm(x_in, initial_state=init)                   # builds the layer
kernel = np.hstack([W[2:] for W in (W_i, W_f, W_c, W_o)])           # (3, 8)
recurrent = np.hstack([W[:2] for W in (W_i, W_f, W_c, W_o)])        # (2, 8)
lstm.set_weights([kernel, recurrent, np.zeros(8)])
_, h_keras, c_keras = lstm(x_in, initial_state=init)
print("Keras h_t:", np.round(np.array(h_keras)[0], 3), " c_t:", np.round(np.array(c_keras)[0], 3))
assert np.allclose(h_keras, h, atol=1e-6) and np.allclose(c_keras, c, atol=1e-6)

In [ ]:
# Keeping the cell state unchanged: f_t = 1 and i_t = 0 copy c_(t-1) straight to c_t
print("f = 1, i = 0:", np.ones(2) * c_prev + np.zeros(2) * c_tilde, " c_prev:", c_prev)

## 2. Parameter count
4 layers (f, i, c~, o), each (units + features) x units weights plus units biases.

In [ ]:
for units, d in [(3, 4), (2, 3)]:
    m = keras.Sequential([keras.Input(shape=(None, d)), keras.layers.LSTM(units)])
    print(f"units={units}, features={d}: Keras {m.count_params()}, formula {4 * ((units + d) * units + units)}")
    assert m.count_params() == 4 * ((units + d) * units + units)
print("SimpleRNN with units=3, features=4:",
      keras.Sequential([keras.Input(shape=(None, 4)), keras.layers.SimpleRNN(3)]).count_params())

## 3. The gates on a real review
A small LSTM (8 units) learns the sentiment of IMDB reviews. Then we run one short test review through it
by hand and record every gate.

In [ ]:
VOCAB, MAXLEN = 10000, 200
(x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
x_tr = keras.utils.pad_sequences(x_tr[:10000], maxlen=MAXLEN)
y_tr = y_tr[:10000].astype("float32")
model = keras.Sequential([keras.Input(shape=(None,)),
                          keras.layers.Embedding(VOCAB, 16),
                          keras.layers.LSTM(8),
                          keras.layers.Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
hist = model.fit(x_tr, y_tr, epochs=3, batch_size=64, validation_split=0.2, verbose=0)
print("validation accuracy per epoch:", np.round(hist.history["val_accuracy"], 3))

In [ ]:
# Decode reviews back to words (indices are shifted by 3: 0 padding, 1 start, 2 unknown)
index = {v + 3: k for k, v in keras.datasets.imdb.get_word_index().items()}
decode = lambda seq: [index.get(k, "?") for k in seq[1:]]          # drop the start marker
E, (K, R, b), (W_y, b_y) = model.layers[0].get_weights()[0], model.layers[1].get_weights(), model.layers[2].get_weights()
u = 8


def lstm_trace(seq):
    """Run the trained LSTM by hand over one review; return the gates, states and running prediction per word."""
    h, c, rows = np.zeros(u), np.zeros(u), []
    for t, k in enumerate(seq):
        z = E[k] @ K + h @ R + b
        i, f, ct, o = sigmoid(z[:u]), sigmoid(z[u:2 * u]), np.tanh(z[2 * u:3 * u]), sigmoid(z[3 * u:])
        c = f * c + i * ct
        h = o * np.tanh(c)
        rows.append(dict(t=t, f=f, i=i, c_tilde=ct, o=o, c=c, h=h, p=sigmoid(h @ W_y + b_y).item()))
    return rows


# Candidates: short test reviews (10-30 words, no unknown words) that the model classifies correctly.
# We pick the one whose running prediction swings the most, so the gates have something to do.
cands = [(k, s) for k, s in enumerate(x_te) if 10 <= len(s) <= 30 and 2 not in s]
scores = []
for k, s in cands:
    ps = [r["p"] for r in lstm_trace(s[1:])]
    if abs(ps[-1] - y_te[k]) < 0.5:
        scores.append((-(max(ps) - min(ps)), k, ps[-1]))
scores.sort()
print(len(cands), "candidates,", len(scores), "classified correctly")
for neg_swing, k, p in scores[:8]:
    print(k, y_te[k], round(p, 3), round(-neg_swing, 3), " ".join(decode(x_te[k])))

In [ ]:
pick = scores[0][1]
seq = x_te[pick][1:]
words = decode(x_te[pick])
trace = lstm_trace(seq)
p_keras = model.predict(np.array([seq]), verbose=0).item()
print("review:", " ".join(words), "| label", y_te[pick])
print("by hand:", round(trace[-1]["p"], 4), " Keras:", round(p_keras, 4))
assert abs(trace[-1]["p"] - p_keras) < 1e-4

In [ ]:
long = []
for r, w in zip(trace, words):
    for g in ("f", "i", "o", "c"):
        for unit in range(u):
            long.append(dict(t=r["t"] + 1, word=w, gate=g, unit=unit + 1, value=r[g][unit]))
pd.DataFrame(long).round(4).to_csv(DATA / "gates_review.csv", index=False)
pd.DataFrame([dict(t=r["t"] + 1, word=w, p=r["p"]) for r, w in zip(trace, words)]).round(4).to_csv(
    DATA / "running_prediction.csv", index=False)

In [ ]:
# Summary numbers for the Note: how open is each gate on average, and the units that change most
g = pd.DataFrame(long)
print(g.groupby("gate").value.describe().round(3))
print(pd.DataFrame([dict(t=r["t"] + 1, word=w, p=round(r["p"], 3)) for r, w in zip(trace, words)]).to_string())